# Negative Poisson's ratio material

This notebook reproduces the density-filtered auxetic microstructure in
Fig. 16 of Xia and Breitkopf (2015) using their energy-based homogenization
method and relaxed objective.

> Xia, L., and Breitkopf, P. "Design of materials using topology
> optimization and energy-based homogenization approach in Matlab."
> *Structural and Multidisciplinary Optimization* 52 (2015): 1229-1241.

In [ ]:
from pathlib import Path

import jax
import jax.numpy as np
import matplotlib.pyplot as plt
import numpy as onp
import scipy.sparse
from IPython.display import Image as DisplayImage
from IPython.display import display
from jax.scipy.signal import convolve2d
from jax_fem import logger
from jax_fem.generate_mesh import Mesh, get_meshio_cell_type, rectangle_mesh
from jax_fem.problem import Problem
from jax_fem.solver import ad_wrapper
from matplotlib.patches import Rectangle
from PIL import Image as PILImage

jax.config.update('jax_enable_x64', True)
logger.setLevel('WARNING')

## 1. Periodic unit cell

The periodic fluctuation displacements on opposite boundaries share the
same reduced degrees of freedom. The fluctuation at the four equivalent
corner nodes is fixed to zero to remove rigid-body translation.

In [ ]:
def periodic_reduction(Nx, Ny):
    """Construct the periodic displacement reduction matrix.

    Parameters
    ----------
    Nx, Ny : int
        Number of elements along the two cell directions.

    Returns
    -------
    scipy.sparse.csr_array
        Matrix mapping reduced periodic degrees of freedom to the full mesh.
    """
    rows = []
    columns = []
    values = []
    num_periodic_nodes = Nx * Ny

    for ix in range(Nx + 1):
        for iy in range(Ny + 1):
            node = ix * (Ny + 1) + iy
            representative = (ix % Nx) * Ny + (iy % Ny)

            if representative == 0:
                continue

            for component in range(2):
                rows.append(2 * node + component)
                columns.append(2 * (representative - 1) + component)
                values.append(1.0)

    return scipy.sparse.csr_array(
        (values, (rows, columns)),
        shape=(
            2 * (Nx + 1) * (Ny + 1),
            2 * (num_periodic_nodes - 1),
        ),
    )


class PeriodicCell(Problem):
    """Linear elastic unit cell with a prescribed macroscopic strain."""

    def custom_init(self, P_mat):
        self.fe = self.fes[0]
        self.P_mat = P_mat

    def get_tensor_map(self):
        def stress(u_grad, density, macro_strain):
            E = 1e-9 + density[0] ** 3 * (1.0 - 1e-9)
            nu = 0.3
            mu = E / (2.0 * (1.0 + nu))
            lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
            lam = 2.0 * mu * lam / (lam + 2.0 * mu)
            strain = macro_strain + 0.5 * (u_grad + u_grad.T)
            return lam * np.trace(strain) * np.eye(2) + 2.0 * mu * strain

        return stress

    def set_params(self, params):
        density, macro_strain = params
        self.internal_vars = [
            np.repeat(density[:, None, :], self.fe.num_quads, axis=1),
            np.broadcast_to(
                macro_strain,
                (self.fe.num_cells, self.fe.num_quads, 2, 2),
            ),
        ]


Nx, Ny = 100, 100
cell_type = get_meshio_cell_type('QUAD4')
meshio_mesh = rectangle_mesh(
    Nx=Nx,
    Ny=Ny,
    domain_x=1.0,
    domain_y=1.0,
)
mesh = Mesh(meshio_mesh.points, meshio_mesh.cells_dict[cell_type])
problem = PeriodicCell(
    mesh,
    vec=2,
    dim=2,
    ele_type='QUAD4',
    additional_info=(periodic_reduction(Nx, Ny),),
)

fwd_pred = ad_wrapper(
    problem,
    solver_options={'linear': True, 'spsolve_solver': {}},
    adjoint_solver_options={'spsolve_solver': {}},
)

macro_strains = np.array([
    [[1.0, 0.0], [0.0, 0.0]],
    [[0.0, 0.0], [0.0, 1.0]],
    [[0.0, 0.5], [0.5, 0.0]],
])

## 2. Energy-based homogenization

For the three unit-strain cases, the homogenized stiffness is evaluated by
the mutual-energy expression

$$
C_{ij}^{H}=\frac{1}{|Y|}\int_Y
\boldsymbol\varepsilon^{A(i)}:\mathbf C(\rho):
\boldsymbol\varepsilon^{A(j)}\,\mathrm dY.
$$

In [ ]:
def homogenized_stiffness(density):
    """Compute the homogenized plane-stress stiffness matrix."""
    total_strains = []
    for macro_strain in macro_strains:
        fluctuation = fwd_pred((density, macro_strain))[0]
        gradient = problem.fe.sol_to_grad(fluctuation)
        total_strains.append(
            macro_strain[None, None]
            + 0.5 * (gradient + np.swapaxes(gradient, 2, 3))
        )
    total_strains = np.stack(total_strains)

    E = 1e-9 + density[:, 0] ** 3 * (1.0 - 1e-9)
    nu = 0.3
    mu = E / (2.0 * (1.0 + nu))
    lam = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
    lam = 2.0 * mu * lam / (lam + 2.0 * mu)

    stresses = []
    for strain in total_strains:
        stresses.append(
            lam[:, None, None, None]
            * np.trace(strain, axis1=2, axis2=3)[:, :, None, None]
            * np.eye(2)
            + 2.0 * mu[:, None, None, None] * strain
        )
    stresses = np.stack(stresses)

    cell_volume = np.sum(problem.fe.JxW)
    return np.einsum(
        'icqab,jcqab,cq->ij',
        total_strains,
        stresses,
        problem.fe.JxW,
    ) / cell_volume


solid_density = np.ones((Nx * Ny, 1))
solid_stiffness = homogenized_stiffness(solid_density)
analytical_stiffness = 1.0 / (1.0 - 0.3**2) * np.array([
    [1.0, 0.3, 0.0],
    [0.3, 1.0, 0.0],
    [0.0, 0.0, (1.0 - 0.3) / 2.0],
])
print('Homogenization check:')
print(onp.asarray(solid_stiffness))
print(
    'Maximum error:',
    float(np.max(np.abs(solid_stiffness - analytical_stiffness))),
)

## 3. Density filter and initial topology

In [ ]:
volume_fraction = 0.5
penal = 3.0
rmin = 5.0
beta = 0.8

radius = int(onp.ceil(rmin)) - 1
offsets = onp.arange(-radius, radius + 1)
dx, dy = onp.meshgrid(offsets, offsets, indexing='xy')
filter_kernel = np.asarray(
    onp.maximum(0.0, rmin - onp.sqrt(dx**2 + dy**2))
)
filter_normalizer = convolve2d(
    np.ones((Ny, Nx)),
    filter_kernel,
    mode='same',
)


def density_filter(density):
    return convolve2d(
        density,
        filter_kernel,
        mode='same',
    ) / filter_normalizer


ix, iy = np.meshgrid(np.arange(Nx), np.arange(Ny), indexing='xy')
design = volume_fraction * np.ones((Ny, Nx))
inside_circle = (
    np.sqrt(
        (ix - Nx / 2.0 + 0.5) ** 2
        + (iy - Ny / 2.0 + 0.5) ** 2
    )
    < min(Nx, Ny) / 6.0
)
design = np.where(inside_circle, volume_fraction / 2.0, design)
physical_density = design.copy()

## 4. Relaxed negative-Poisson-ratio objective

At iteration $l$, the objective proposed in the paper is

$$
c^{(l)}=C_{12}^{H}-0.8^l\left(C_{11}^{H}+C_{22}^{H}\right).
$$

In [ ]:
def objective(physical_density, iteration):
    density = physical_density.T.reshape(-1, 1)
    stiffness = homogenized_stiffness(density)
    value = stiffness[0, 1] - beta**iteration * (
        stiffness[0, 0] + stiffness[1, 1]
    )
    return value, stiffness


def filtered_sensitivity(sensitivity):
    return convolve2d(
        sensitivity / filter_normalizer,
        filter_kernel,
        mode='same',
    )


def volume(design):
    return np.mean(density_filter(design))


def oc_update(design, sensitivity, volume_gradient):
    lower_lagrange = 0.0
    upper_lagrange = 1e9
    move = 0.1

    while upper_lagrange - lower_lagrange > 1e-9:
        lagrange = 0.5 * (lower_lagrange + upper_lagrange)
        trial = np.maximum(
            0.0,
            np.maximum(
                design - move,
                np.minimum(
                    1.0,
                    np.minimum(
                        design + move,
                        design
                        * (-sensitivity / volume_gradient / lagrange),
                    ),
                ),
            ),
        )
        if volume(trial) > volume_fraction:
            lower_lagrange = lagrange
        else:
            upper_lagrange = lagrange

    return trial

## 5. Gradient check

In [ ]:
def filtered_objective(raw_density):
    return objective(density_filter(raw_density), 1)[0]


_, objective_gradient = jax.value_and_grad(filtered_objective)(design)
direction = np.sin(np.arange(design.size)).reshape(design.shape)
direction = direction / np.linalg.norm(direction)
epsilon = 1e-4
finite_difference = (
    filtered_objective(design + epsilon * direction)
    - filtered_objective(design - epsilon * direction)
) / (2.0 * epsilon)
automatic_derivative = np.sum(objective_gradient * direction)
relative_error = np.abs(
    finite_difference - automatic_derivative
) / np.maximum(np.abs(finite_difference), 1e-12)

print(f'Finite difference: {float(finite_difference):.8e}')
print(f'AD derivative:     {float(automatic_derivative):.8e}')
print(f'Relative error:    {float(relative_error):.3e}')

## 6. Optimization

In [ ]:
volume_gradient = filtered_sensitivity(np.ones((Ny, Nx))) / (Nx * Ny)
change = 1.0
iteration = 0
records = []
frames = []

while change > 0.01 and iteration < 250:
    iteration += 1
    (value, stiffness), sensitivity = jax.value_and_grad(
        objective,
        has_aux=True,
    )(physical_density, iteration)
    sensitivity = filtered_sensitivity(sensitivity)

    old_design = design.copy()
    design = oc_update(old_design, sensitivity, volume_gradient)
    physical_density = density_filter(design)
    change = float(np.max(np.abs(design - old_design)))
    poisson_ratio = float(stiffness[0, 1] / stiffness[0, 0])

    record = {
        'iteration': iteration,
        'objective': float(value),
        'volume': float(np.mean(physical_density)),
        'change': change,
        'poisson_ratio': poisson_ratio,
        'stiffness': onp.asarray(stiffness),
    }
    records.append(record)

    if iteration == 1 or iteration % 2 == 0:
        frames.append({
            **record,
            'density': onp.asarray(physical_density),
        })

    print(
        f"It.:{iteration:4d}, Obj.:{float(value):11.6f}, "
        f"Vol.:{record['volume']:7.4f}, "
        f"nu:{poisson_ratio:8.4f}, ch.:{change:7.4f}"
    )


final_density = physical_density.T.reshape(-1, 1)
final_stiffness = homogenized_stiffness(final_density)
final_poisson_ratio = float(final_stiffness[0, 1] / final_stiffness[0, 0])
final_record = {
    'iteration': iteration,
    'objective': float(final_stiffness[0, 1]),
    'volume': float(np.mean(physical_density)),
    'change': change,
    'poisson_ratio': final_poisson_ratio,
    'stiffness': onp.asarray(final_stiffness),
    'density': onp.asarray(physical_density),
}
frames.append(final_record)

## 7. Save data and animation

In [ ]:
output_data = Path('docs/data/example_topopt_auxetic.npz')
output_data.parent.mkdir(parents=True, exist_ok=True)
onp.savez_compressed(
    output_data,
    design=onp.asarray(design),
    physical_density=onp.asarray(physical_density),
    homogenized_stiffness=onp.asarray(final_stiffness),
    frame_iteration=onp.array([frame['iteration'] for frame in frames]),
    frame_density=onp.stack([frame['density'] for frame in frames]),
    frame_objective=onp.array([frame['objective'] for frame in frames]),
    frame_volume=onp.array([frame['volume'] for frame in frames]),
    frame_poisson_ratio=onp.array(
        [frame['poisson_ratio'] for frame in frames]
    ),
    record_iteration=onp.array([record['iteration'] for record in records]),
    record_objective=onp.array([record['objective'] for record in records]),
    record_volume=onp.array([record['volume'] for record in records]),
    record_change=onp.array([record['change'] for record in records]),
    record_poisson_ratio=onp.array(
        [record['poisson_ratio'] for record in records]
    ),
    record_stiffness=onp.stack([record['stiffness'] for record in records]),
)


def render_frame(frame):
    with plt.rc_context({
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 21,
    }):
        fig, ax = plt.subplots(figsize=(7, 7.4), dpi=150)
        ax.imshow(
            frame['density'],
            origin='lower',
            cmap='gray_r',
            vmin=0.0,
            vmax=1.0,
            interpolation='nearest',
        )
        ax.set_aspect('equal')
        ax.set_axis_off()
        ax.set_title(
            rf"$i={frame['iteration']}\quad c={frame['objective']:.5f}$"
            '\n'
            rf"$V/V_0={100.0 * frame['volume']:.1f}\%\quad "
            rf"\nu^H={frame['poisson_ratio']:.3f}$",
            fontsize=23,
            pad=12,
        )
        fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.84)
        fig.canvas.draw()
        rgba = onp.asarray(fig.canvas.buffer_rgba()).copy()
        plt.close(fig)
    return PILImage.fromarray(rgba)


images = [render_frame(frame) for frame in frames]
gif_path = Path('docs/imgs/example_topopt_auxetic.gif')
gif_path.parent.mkdir(parents=True, exist_ok=True)
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=140,
    loop=0,
    optimize=False,
)
display(DisplayImage(filename=str(gif_path)))

print('Final homogenized stiffness:')
print(onp.asarray(final_stiffness))
print(f'Final Poisson ratio: {final_poisson_ratio:.6f}')
print(f'Final volume fraction: {float(np.mean(physical_density)):.6f}')
print(f'Iterations: {iteration}')

## 8. Uniaxial deformation demonstration

The transverse and engineering shear strains are chosen so that the
averaged transverse and shear stresses vanish during horizontal tension.
A periodic `3 × 3` array is shown to make the auxetic expansion visible.

In [ ]:
free_strains = -np.linalg.solve(
    final_stiffness[1:, 1:],
    final_stiffness[1:, 0],
)
macro_strain_vector = np.array([1.0, free_strains[0], free_strains[1]])
uniaxial_macro_strain = np.array([
    [macro_strain_vector[0], macro_strain_vector[2] / 2.0],
    [macro_strain_vector[2] / 2.0, macro_strain_vector[1]],
])
uniaxial_fluctuation = fwd_pred(
    (final_density, uniaxial_macro_strain)
)[0]
uniaxial_stress = final_stiffness @ macro_strain_vector

print('Unit uniaxial-strain vector:', onp.asarray(macro_strain_vector))
print('Homogenized stress:', onp.asarray(uniaxial_stress))

tiles = 3
maximum_axial_strain = 0.15
loading_factors = onp.concatenate((
    onp.linspace(0.0, 1.0, 21),
    onp.linspace(0.95, 0.0, 20),
))
grid_x, grid_y = onp.meshgrid(
    onp.linspace(0.0, tiles, tiles * Nx + 1),
    onp.linspace(0.0, tiles, tiles * Ny + 1),
    indexing='xy',
)
fluctuation_x = onp.asarray(uniaxial_fluctuation[:, 0]).reshape(
    Nx + 1, Ny + 1
).T
fluctuation_y = onp.asarray(uniaxial_fluctuation[:, 1]).reshape(
    Nx + 1, Ny + 1
).T
periodic_x = onp.tile(fluctuation_x[:-1, :-1], (tiles, tiles))
periodic_y = onp.tile(fluctuation_y[:-1, :-1], (tiles, tiles))
periodic_x = onp.pad(periodic_x, ((0, 1), (0, 1)), mode='wrap')
periodic_y = onp.pad(periodic_y, ((0, 1), (0, 1)), mode='wrap')
tiled_density = onp.tile(onp.asarray(physical_density), (tiles, tiles))

center = onp.array([tiles / 2.0, tiles / 2.0])
relative_x = grid_x - center[0]
relative_y = grid_y - center[1]
total_x = (
    float(uniaxial_macro_strain[0, 0]) * relative_x
    + float(uniaxial_macro_strain[0, 1]) * relative_y
    + periodic_x
)
total_y = (
    float(uniaxial_macro_strain[1, 0]) * relative_x
    + float(uniaxial_macro_strain[1, 1]) * relative_y
    + periodic_y
)
final_x = grid_x + maximum_axial_strain * total_x
final_y = grid_y + maximum_axial_strain * total_y
xmin = min(onp.min(grid_x), onp.min(final_x))
xmax = max(onp.max(grid_x), onp.max(final_x))
ymin = min(onp.min(grid_y), onp.min(final_y))
ymax = max(onp.max(grid_y), onp.max(final_y))
margin = 0.04 * max(xmax - xmin, ymax - ymin)
bounds = (xmin - margin, xmax + margin, ymin - margin, ymax + margin)


def render_deformation(load_factor):
    axial_strain = load_factor * maximum_axial_strain
    transverse_strain = axial_strain * float(macro_strain_vector[1])
    deformed_x = grid_x + axial_strain * total_x
    deformed_y = grid_y + axial_strain * total_y

    with plt.rc_context({
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 20,
    }):
        fig, ax = plt.subplots(figsize=(7.4, 7.8), dpi=150)
        ax.add_patch(Rectangle(
            (0.0, 0.0),
            tiles,
            tiles,
            fill=False,
            edgecolor='0.72',
            linewidth=1.0,
            linestyle='--',
        ))
        ax.pcolormesh(
            deformed_x,
            deformed_y,
            tiled_density,
            cmap='gray_r',
            vmin=0.0,
            vmax=1.0,
            shading='flat',
            rasterized=True,
        )
        ax.set_xlim(bounds[0], bounds[1])
        ax.set_ylim(bounds[2], bounds[3])
        ax.set_aspect('equal')
        ax.set_axis_off()
        ax.set_title(
            rf'$\bar{{\varepsilon}}_{{xx}}={100.0 * axial_strain:.1f}\%$'
            '\n'
            rf'$\bar{{\varepsilon}}_{{yy}}='
            rf'{100.0 * transverse_strain:.1f}\%\quad '
            rf'\nu^H={final_poisson_ratio:.3f}$',
            fontsize=23,
            pad=10,
        )
        fig.subplots_adjust(left=0.01, right=0.99, bottom=0.01, top=0.86)
        fig.canvas.draw()
        rgba = onp.asarray(fig.canvas.buffer_rgba()).copy()
        plt.close(fig)
    return PILImage.fromarray(rgba)


deformation_images = [
    render_deformation(load_factor) for load_factor in loading_factors
]
deformation_path = Path('docs/imgs/example_topopt_auxetic_deformation.gif')
deformation_images[0].save(
    deformation_path,
    save_all=True,
    append_images=deformation_images[1:],
    duration=100,
    loop=0,
    optimize=False,
)
display(DisplayImage(filename=str(deformation_path)))